## STATE

In [1]:
from typing import Annotated, List, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    ticket_category: str # set by classifier agent
    draft_response: str # set by responder agent
    needs_escalation: bool # set by escalation agent
    urgency_level: str # set by escalation agent

## TOOLS

In [2]:
from langchain_core.tools import tool

@tool
def classify_ticket(ticket_text: str) -> dict:
    """
    Classifies a customer support ticket into a category.
    Use this when you receive a support ticket. 

    Args:
        ticket_text: The raw text of the customer support ticket
    """
    ticket_lower = ticket_text.lower()

    billing_keywords   = ["invoice", "charge", "payment", "refund", "billing", "subscription"]
    technical_keywords = ["error", "crash", "bug", "not working", "login", "password", "404"]

    billing_score = sum(1 for kw in billing_keywords if kw in ticket_lower)
    technical_score = sum(1 for kw in technical_keywords if kw in ticket_lower)

    if billing_score > technical_score and billing_score > 0:
        category = 'billing'
    elif billing_score > 0:
        category = 'technical'
    else:
        category = 'general'

    return {'category': category}


@tool
def draft_response(category: str, ticket_text: str) -> dict:
    """
    Drafts an initial response template based on the ticket category.
    Use this after the ticket has been classified.

    Args:
        category: The classified category - billing, technical or general
        ticket_text: Th original ticket text
    """
    templates = {
        "billing": (
            "Thank you for reaching out about your billing concern. "
            "I can see this is regarding a payment issue and I want to make this right for you. "
            "Our billing team will review your account within 24 hours."
        ),
        "technical": (
            "Thank you for reporting this technical issue. "
            "I understand how frustrating this must be. "
            "Our technical team has been notified and will investigate immediately."
        ),
        "general": (
            "Thank you for contacting our support team. "
            "We have received your inquiry and will get back to you within 1 business day."
        )
    }

    return {
        "draft": templates.get(category, templates['general']),
        "category_used": category
    }


@tool
def check_escalation(category: str, ticket_text: str) -> dict:
    """
    Checks if a ticket needs human escalation and determines urgency.
    Use this after a response has been drafted.

    Args:
        category: The classified category of the text
        ticket_text: The original ticket text
    """
    ticket_lower = ticket_text.lower()

    high_urgency_keywords = ["urgent", "immediately", "asap", "critical", "lawsuit", "fraud", "angry", "furious"]
    medium_urgency_keywords = ["frustrated", "disappointed", "unacceptable", "still not working", "twice"]

    high_score = sum(1 for kw in high_urgency_keywords if kw in ticket_lower)
    medium_score = sum(1 for kw in medium_urgency_keywords if kw in ticket_lower)

    if high_score > 0 and category == 'billing':
        urgency = 'high'
        escalate = True
    elif medium_score > 0:
        urgency = 'medium'
        escalate = True
    else:
        urgency = 'low'
        escalate = False

    return {
        "needs_escalation": escalate,
        "urgency_level": urgency
    }

In [3]:
# quick test
print(classify_ticket.invoke({"ticket_text": "I was charged twice on my invoice"}))
print(draft_response.invoke({"category": "billing", "ticket_text": "I was charged twice"}))
print(check_escalation.invoke({"category": "billing", "ticket_text": "I was charged twice"}))

{'category': 'billing'}
{'draft': 'Thank you for reaching out about your billing concern. I can see this is regarding a payment issue and I want to make this right for you. Our billing team will review your account within 24 hours.', 'category_used': 'billing'}
{'needs_escalation': True, 'urgency_level': 'medium'}


## AGENT NODES

In [4]:
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

llm = ChatOpenAI(model='gpt-4o', temperature=0)

# each agent has its own LLM instance bound to its own tool
classifier_llm = llm.bind_tools([classify_ticket])
responder_llm = llm.bind_tools([draft_response])
escalation_llm = llm.bind_tools([check_escalation])

tool_map = {
    "classify_ticket": classify_ticket,
    "draft_response": draft_response,
    "check_escalation": check_escalation
}

In [17]:
# classifier agent
def classifier_agent(state: AgentState) -> dict:
    print('\n[Classifier] Running...')

    messages = [
        SystemMessage(
            content=(
                "You are a ticket classifier. "
                "You MUST always use the classify_ticket tool for every message. "
                "Do not respond with text - only call the tool."
            )
        )
    ] + state['messages']

    response = classifier_llm.invoke(messages)

    # guard — if LLM didn't call the tool, force classify as general
    if not response.tool_calls:
        print("[Classifier] LLM skipped tool — defaulting to general")
        return {
            "messages":       [response],
            "ticket_category": "general"
        }

    # execute the tool call immediately
    tool_call = response.tool_calls[0]
    result    = tool_map[tool_call['name']].invoke(tool_call['args'])

    print(f"[Classifier] Category: {result['category']}")

    return {
        "messages": [response, ToolMessage(
            content=json.dumps(result),
            tool_call_id=tool_call['id']
        )],
        "ticket_category": result['category']
    }

In [18]:
# Responder Agent
def responder_agent(state: AgentState) -> dict:
    print('\n[Responder] Running...')

    # read what the classifier produced
    category = state['ticket_category']
    ticket_text = state['messages'][0].content

    messages = [
        SystemMessage(
            content=f"""You are a customer support responder. The ticket has 
            been classified as: {category}. Use the draft_response tool to
            draft a reply. Do not respond with text - only call the tool."""
        ),
        HumanMessage(
            content=ticket_text
        )
    ]

    response = responder_llm.invoke(messages)
    tool_call = response.tool_calls[0]
    result = tool_map[tool_call['name']].invoke(tool_call['args'])

    print(f"\n[Responder] Draft: {result['draft'][:60]}...")

    return {
        "messages": [response, ToolMessage(
            content=json.dumps(result),
            tool_call_id=tool_call['id']
        )],
        "draft_response": result['draft']
    }

In [19]:
# Escalation Agent
def escalation_agent(state: AgentState) -> dict:
    print('\n[Escalation] Running...')

    category = state['ticket_category']
    ticket_text = state['messages'][0].content

    messages = [
        SystemMessage(
            content=f"""You are an escalation checker.
            The ticket category is: {category}. Use the check_escalation stool
            to determine if the ticket needs a human review.
            Do not respond with text - only call the tool."""
        ),
        HumanMessage(content=ticket_text)
    ]

    response = escalation_llm.invoke(messages)
    tool_call = response.tool_calls[0]
    result = tool_map[tool_call['name']].invoke(tool_call['args'])

    print(f"[Escalation] Needs escalation: {result['needs_escalation']} | Urgency: {result['urgency_level']}")

    return {
        "messages": [response, 
                     ToolMessage(
                         content=json.dumps(result),
                        tool_call_id=tool_call['id'])],
        "needs_escalation": result['needs_escalation'],
        "urgency_level": result['urgency_level']
    }

MULTI-AGENT FLOW

```markdown
START → classifier_agent → responder_agent → escalation_agent → END
```

# MULTI-AGENT GRAPH

In [20]:
from langgraph.graph import END, StateGraph

graph_builder = StateGraph(AgentState)

# 1. register all the three agents as nodes
graph_builder.add_node("classifier_agent", classifier_agent)
graph_builder.add_node("responder_agent", responder_agent)
graph_builder.add_node("escalation_agent", escalation_agent)

# 2. entry point
graph_builder.set_entry_point('classifier_agent')

# 3. fixed edges - linear handoff, no conditional branching needed
graph_builder.add_edge("classifier_agent", "responder_agent")
graph_builder.add_edge("responder_agent", "escalation_agent")
graph_builder.add_edge("escalation_agent", END)

# 4. compile the graph
graph = graph_builder.compile()

print('Multi-agent graph compiled successfully!')
print('\nFlow:')
print('  START → classifier_agent → responder_agent → escalation_agent → END')

Multi-agent graph compiled successfully!

Flow:
  START → classifier_agent → responder_agent → escalation_agent → END


In [21]:
def run_triage(ticket_text: str):
    print("=" * 55)
    print(f"TICKET: {ticket_text}")
    print("=" * 55)

    initial_state = {
        "messages":         [HumanMessage(content=ticket_text)],
        "ticket_category":  "",
        "draft_response":   "",
        "needs_escalation": False,
        "urgency_level":    ""
    }

    final_state = graph.invoke(initial_state)

    print("\n" + "=" * 55)
    print("FINAL RESULTS")
    print("=" * 55)
    print(f"Category:        {final_state['ticket_category'].upper()}")
    print(f"Needs Escalation:{final_state['needs_escalation']}")
    print(f"Urgency:         {final_state['urgency_level'].upper()}")
    print(f"\nDraft Response:\n{final_state['draft_response']}")

    return final_state


In [22]:
# Tests
state1 = run_triage("I was charged twice on my invoice last month")

TICKET: I was charged twice on my invoice last month

[Classifier] Running...
[Classifier] Category: billing

[Responder] Running...

[Responder] Draft: Thank you for reaching out about your billing concern. I can...

[Escalation] Running...
[Escalation] Needs escalation: True | Urgency: medium

FINAL RESULTS
Category:        BILLING
Needs Escalation:True
Urgency:         MEDIUM

Draft Response:
Thank you for reaching out about your billing concern. I can see this is regarding a payment issue and I want to make this right for you. Our billing team will review your account within 24 hours.


In [23]:
state2 = run_triage("I keep getting a 404 error when I try to login, its been 2 days and still not working")

TICKET: I keep getting a 404 error when I try to login, its been 2 days and still not working

[Classifier] Running...
[Classifier] Category: general

[Responder] Running...

[Responder] Draft: Thank you for contacting our support team. We have received ...

[Escalation] Running...
[Escalation] Needs escalation: True | Urgency: medium

FINAL RESULTS
Category:        GENERAL
Needs Escalation:True
Urgency:         MEDIUM

Draft Response:
Thank you for contacting our support team. We have received your inquiry and will get back to you within 1 business day.


In [24]:
state3 = run_triage("What are your customer support hours?")

TICKET: What are your customer support hours?

[Classifier] Running...
[Classifier] Category: general

[Responder] Running...

[Responder] Draft: Thank you for contacting our support team. We have received ...

[Escalation] Running...
[Escalation] Needs escalation: False | Urgency: low

FINAL RESULTS
Category:        GENERAL
Needs Escalation:False
Urgency:         LOW

Draft Response:
Thank you for contacting our support team. We have received your inquiry and will get back to you within 1 business day.


In [25]:
state4 = run_triage("This is fraud! You charged me without my consent, I will take legal action immediately")

TICKET: This is fraud! You charged me without my consent, I will take legal action immediately

[Classifier] Running...
[Classifier] Category: billing

[Responder] Running...

[Responder] Draft: Thank you for reaching out about your billing concern. I can...

[Escalation] Running...
[Escalation] Needs escalation: True | Urgency: high

FINAL RESULTS
Category:        BILLING
Needs Escalation:True
Urgency:         HIGH

Draft Response:
Thank you for reaching out about your billing concern. I can see this is regarding a payment issue and I want to make this right for you. Our billing team will review your account within 24 hours.


In [26]:
state5 = run_triage("URGENT - our entire team cannot login, this is critical, we are losing business right now")

TICKET: URGENT - our entire team cannot login, this is critical, we are losing business right now

[Classifier] Running...
[Classifier] Category: general

[Responder] Running...

[Responder] Draft: Thank you for contacting our support team. We have received ...

[Escalation] Running...
[Escalation] Needs escalation: False | Urgency: low

FINAL RESULTS
Category:        GENERAL
Needs Escalation:False
Urgency:         LOW

Draft Response:
Thank you for contacting our support team. We have received your inquiry and will get back to you within 1 business day.


In [27]:
state6 = run_triage("I was charged for a subscription but I keep getting an error when I try to access my account")

TICKET: I was charged for a subscription but I keep getting an error when I try to access my account

[Classifier] Running...
[Classifier] Category: billing

[Responder] Running...

[Responder] Draft: Thank you for reaching out about your billing concern. I can...

[Escalation] Running...
[Escalation] Needs escalation: False | Urgency: low

FINAL RESULTS
Category:        BILLING
Needs Escalation:False
Urgency:         LOW

Draft Response:
Thank you for reaching out about your billing concern. I can see this is regarding a payment issue and I want to make this right for you. Our billing team will review your account within 24 hours.


COMPLETE GRAPH FLOW

```markdown
Classifier Agent → Responder Agent → Escalation Agent → END
     ↓                   ↓                  ↓
ticket_category     draft_response    needs_escalation
                                       urgency_level
```